In [1]:
from pathlib import Path
import sys

# ============================================================
# ElShaddAI — Module 5
# Project Integration & Validation
# ============================================================

PROJECT_ROOT = Path(r"C:\Users\acer\ElShaddAI")

# Make project root importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SRC_DIR = PROJECT_ROOT / "src"
MODELS_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "data"
FRONTEND_DIR = PROJECT_ROOT / "frontend"
REACT_FRONTEND_DIR = PROJECT_ROOT / "frontend-react"

print("Project root:")
print(PROJECT_ROOT)

print("\nPython executable:")
print(sys.executable)

print("\nProject directories:")
print("src:", SRC_DIR.exists())
print("models:", MODELS_DIR.exists())
print("data:", DATA_DIR.exists())
print("frontend:", FRONTEND_DIR.exists())
print("frontend-react:", REACT_FRONTEND_DIR.exists())

print("\nPython version:")
print(sys.version)

Project root:
C:\Users\acer\ElShaddAI

Python executable:
C:\Users\acer\anaconda3\envs\project_antarctica\python.exe

Project directories:
src: True
models: True
data: True
frontend: True
frontend-react: False

Python version:
3.14.7 | packaged by Anaconda, Inc. | (main, Aug 14 2026, 10:12:46) [MSC v.1942 64 bit (AMD64)]


In [2]:
# ============================================================
# Module Import Health Check
# ============================================================

modules_to_test = {
    "Module 1 - Forecast": "src.module1_forecast",
    "Module 1 - Risk": "src.module1_risk",
    "Module 1 - Cost": "src.module1_cost",
    "Module 1 - Route": "src.module1_route",
    "Module 2 - Trajectory": "src.module2_trajectory",
    "Module 2 - Risk": "src.module2_risk",
    "Module 3 - Decision": "src.module3_decision",
    "Module 4 - Navigation": "src.module4_navigation",
}

import importlib

results = {}

for name, module_name in modules_to_test.items():
    try:
        importlib.import_module(module_name)
        results[name] = "PASS"
    except Exception as exc:
        results[name] = f"FAIL: {type(exc).__name__}: {exc}"

print("=" * 70)
print("ELSHADDAI MODULE HEALTH CHECK")
print("=" * 70)

for name, result in results.items():
    print(f"{name:<35} {result}")

failed = [
    name for name, result in results.items()
    if result != "PASS"
]

print("\n" + "=" * 70)

if not failed:
    print("ALL PRODUCTION MODULE IMPORTS: PASS")
else:
    print("FAILED MODULES:")
    for name in failed:
        print(" -", name)

ELSHADDAI MODULE HEALTH CHECK
Module 1 - Forecast                 PASS
Module 1 - Risk                     PASS
Module 1 - Cost                     PASS
Module 1 - Route                    PASS
Module 2 - Trajectory               PASS
Module 2 - Risk                     PASS
Module 3 - Decision                 PASS
Module 4 - Navigation               PASS

ALL PRODUCTION MODULE IMPORTS: PASS


In [3]:
# ============================================================
# End-to-End Navigation Pipeline Test
# ============================================================

from src.module1_forecast import forecast_sic
from src.module1_risk import classify_sic_risk
from src.vessel_profiles import create_vessel_cost_surface

from src.module2_trajectory import (
    get_coverage_aware_predictions
)

from src.module2_risk import (
    create_weighted_iceberg_cost_surface
)

from src.module3_decision import (
    create_integrated_navigation_cost
)

from src.module4_navigation import (
    plan_navigation_route
)

import numpy as np


# ------------------------------------------------------------
# Test configuration
# ------------------------------------------------------------

FORECAST_DATE = "2025-09-15"

START_LAT = -59.9258156
START_LON = 49.8904037

DEST_LAT = -59.9328194
DEST_LON = 68.5594406

VESSEL_PROFILE = "standard"


print("=" * 70)
print("ELSHADDAI END-TO-END PIPELINE TEST")
print("=" * 70)

print("\n1. Module 1 — Sea-Ice Forecast")

forecast = forecast_sic(
    FORECAST_DATE
)

predicted_sic = forecast["predicted_sic"]

risk_code = classify_sic_risk(
    predicted_sic
)

sea_ice_cost = create_vessel_cost_surface(
    risk_code,
    VESSEL_PROFILE
)

print("   Forecast date:", FORECAST_DATE)
print("   Prediction date:",
      forecast["prediction_date"])
print("   Grid shape:",
      predicted_sic.shape)

print("\n2. Module 2 — Iceberg Prediction")

coverage = get_coverage_aware_predictions(
    FORECAST_DATE,
    max_persistence_days=3,
    persistence_hazard_weight=0.25
)

weighted_iceberg = create_weighted_iceberg_cost_surface(
    forecast["latitude"],
    forecast["longitude"],
    coverage["predictions"],
    influence_km=30.0,
    hard_avoid_km=5.0,
    max_penalty=500.0
)

iceberg_cost = weighted_iceberg["iceberg_cost"]

print("   Total tracks:",
      coverage["coverage"]["total_tracks"])

print("   Represented tracks:",
      coverage["coverage"]["represented_tracks"])

print("   Coverage:",
      coverage["coverage"]["coverage_percent"], "%")


print("\n3. Module 3 — Decision Fusion")

navigation_cost = create_integrated_navigation_cost(
    sea_ice_cost=sea_ice_cost,
    iceberg_cost=iceberg_cost,
    iceberg_weight=1.0
)

print("   Navigation grid:",
      navigation_cost.shape)

print("   Valid cells:",
      np.isfinite(navigation_cost).sum())


print("\n4. Module 4 — Navigation")

spatial_output = {
    "predicted_sic": predicted_sic,
    "risk_code": risk_code,
    "latitude": forecast["latitude"],
    "longitude": forecast["longitude"],
    "yc": forecast["yc"],
    "xc": forecast["xc"],
}

route = plan_navigation_route(
    start_latitude=START_LAT,
    start_longitude=START_LON,
    destination_latitude=DEST_LAT,
    destination_longitude=DEST_LON,
    navigation_cost=navigation_cost,
    spatial_output=spatial_output
)

print("   Navigation mode:",
      route["navigation_mode"])

print("   Route cells:",
      route["route"]["grid_cells"])

print("   Distance:",
      route["route"]["distance_km"],
      "km")

print("   Straight-line:",
      route["route"]["straight_line_distance_km"],
      "km")

print("   Detour:",
      route["route"]["detour_factor"])

print("   Navigation cost:",
      route["route"]["total_navigation_cost"])


print("\n" + "=" * 70)
print("END-TO-END PIPELINE: PASS")
print("=" * 70)

ELSHADDAI END-TO-END PIPELINE TEST

1. Module 1 — Sea-Ice Forecast
   Forecast date: 2025-09-15
   Prediction date: 2025-09-16 00:00:00
   Grid shape: (432, 432)

2. Module 2 — Iceberg Prediction
   Total tracks: 127
   Represented tracks: 31
   Coverage: 24.41 %

3. Module 3 — Decision Fusion
   Navigation grid: (432, 432)
   Valid cells: 153319

4. Module 4 — Navigation
   Navigation mode: initial_route
   Route cells: 38
   Distance: 1152.82 km
   Straight-line: 1076.16 km
   Detour: 1.071
   Navigation cost: 46.11

END-TO-END PIPELINE: PASS


In [4]:
import requests

url = "http://127.0.0.1:8000/iceberg-risk-grid"

payload = {
    "forecast_date": "2025-09-15"
}

response = requests.post(
    url,
    json=payload,
    timeout=120
)

print("HTTP status:", response.status_code)

if response.ok:
    data = response.json()

    print("\nTop-level keys:")
    print(list(data.keys()))

    print("\nData types:")
    for key, value in data.items():
        print(
            f"{key}:",
            type(value).__name__
        )

    # Inspect likely iceberg prediction containers
    for key in [
        "predictions",
        "points",
        "iceberg_predictions",
        "iceberg_coverage",
        "coverage"
    ]:
        if key in data:
            value = data[key]

            print(f"\n--- {key} ---")

            if isinstance(value, list):
                print("Count:", len(value))

                if value:
                    print("First item:")
                    print(value[0])

            elif isinstance(value, dict):
                print("Keys:")
                print(list(value.keys()))
                print("Value:")
                print(value)

            else:
                print(value)

else:
    print(response.text)

HTTP status: 200

Top-level keys:
['status', 'forecast_date', 'prediction_date', 'grid_step', 'iceberg_coverage', 'cost_parameters', 'predicted_icebergs', 'points']

Data types:
status: str
forecast_date: str
prediction_date: str
grid_step: int
iceberg_coverage: dict
cost_parameters: dict
predicted_icebergs: list
points: list

--- points ---
Count: 4651
First item:
{'latitude': -55.10918426513672, 'longitude': -8.817195892333984, 'iceberg_cost': 0.0, 'nearest_iceberg_distance_km': 1859.42724609375, 'nearest_iceberg_weight': 1.0, 'nearest_iceberg_id': 'a23a'}

--- iceberg_coverage ---
Keys:
['total_tracks', 'ml_predictions', 'persistence_estimates', 'represented_tracks', 'excluded_tracks', 'coverage_percent']
Value:
{'total_tracks': 127, 'ml_predictions': 7, 'persistence_estimates': 24, 'represented_tracks': 31, 'excluded_tracks': 96, 'coverage_percent': 24.41}


In [5]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/replan",
    json={
        "forecast_date": "2025-09-20",
        "vessel_profile": "standard",
        "current_latitude": -59.79476547241211,
        "current_longitude": 60.06848907470703,
        "destination_latitude": -59.9328194,
        "destination_longitude": 68.5594406
    },
    timeout=120
)

print("HTTP status:", response.status_code)

data = response.json()

print("\nDecision:")
print(data.get("decision"))

HTTP status: 200

Decision:
{'route_cells': 21, 'dominant_hazard': 'sea_ice', 'mean_cost': {'sea_ice': 2.0952380952380953, 'iceberg': 0.0, 'combined': 2.0952380952380953}, 'maximum_cost': {'sea_ice': 20.0, 'iceberg': 0.0}, 'iceberg_exposure': {'affected_route_cells': 0, 'maximum_penalty': 0.0}, 'contribution_percent': {'sea_ice': 100.0, 'iceberg': 0.0}, 'iceberg_affected_route_cells': 0}


In [6]:
# ============================================================
# API Performance Baseline
# ============================================================

import requests
import time
import statistics

API_BASE = "http://127.0.0.1:8000"

TEST_DATE = "2025-09-15"

route_payload = {
    "forecast_date": TEST_DATE,
    "vessel_profile": "standard",
    "start_latitude": -59.9258156,
    "start_longitude": 49.8904037,
    "destination_latitude": -59.9328194,
    "destination_longitude": 68.5594406,
}

replan_payload = {
    "forecast_date": "2025-09-20",
    "vessel_profile": "standard",
    "current_latitude": -59.79476547241211,
    "current_longitude": 60.06848907470703,
    "destination_latitude": -59.9328194,
    "destination_longitude": 68.5594406,
}


def measure_request(
    method,
    url,
    json=None,
    runs=3,
):
    timings = []
    statuses = []

    for _ in range(runs):

        start = time.perf_counter()

        response = requests.request(
            method,
            url,
            json=json,
            timeout=180,
        )

        elapsed = (
            time.perf_counter() - start
        )

        timings.append(elapsed)
        statuses.append(response.status_code)

    return {
        "mean_seconds": statistics.mean(timings),
        "min_seconds": min(timings),
        "max_seconds": max(timings),
        "statuses": statuses,
    }


print("=" * 70)
print("ELSHADDAI API PERFORMANCE BASELINE")
print("=" * 70)


tests = {
    "Health": (
        "GET",
        f"{API_BASE}/health",
        None,
    ),

    "Forecast Grid": (
        "POST",
        f"{API_BASE}/forecast-grid",
        {"forecast_date": TEST_DATE},
    ),

    "Iceberg Risk Grid": (
        "POST",
        f"{API_BASE}/iceberg-risk-grid",
        {"forecast_date": TEST_DATE},
    ),

    "Route": (
        "POST",
        f"{API_BASE}/route",
        route_payload,
    ),

    "Replan": (
        "POST",
        f"{API_BASE}/replan",
        replan_payload,
    ),
}


results = {}

for name, (
    method,
    url,
    payload,
) in tests.items():

    result = measure_request(
        method,
        url,
        json=payload,
        runs=3,
    )

    results[name] = result

    print(f"\n{name}")
    print(
        f"  Mean: {result['mean_seconds']:.3f} s"
    )
    print(
        f"  Min : {result['min_seconds']:.3f} s"
    )
    print(
        f"  Max : {result['max_seconds']:.3f} s"
    )
    print(
        f"  HTTP: {result['statuses']}"
    )


print("\n" + "=" * 70)
print("API PERFORMANCE BASELINE COMPLETE")
print("=" * 70)

ELSHADDAI API PERFORMANCE BASELINE

Health
  Mean: 0.022 s
  Min : 0.008 s
  Max : 0.031 s
  HTTP: [200, 200, 200]

Forecast Grid
  Mean: 4.279 s
  Min : 3.835 s
  Max : 5.036 s
  HTTP: [200, 200, 200]

Iceberg Risk Grid
  Mean: 10.648 s
  Min : 10.303 s
  Max : 11.111 s
  HTTP: [200, 200, 200]

Route
  Mean: 10.923 s
  Min : 10.411 s
  Max : 11.589 s
  HTTP: [200, 200, 200]

Replan
  Mean: 12.504 s
  Min : 11.591 s
  Max : 12.973 s
  HTTP: [200, 200, 200]

API PERFORMANCE BASELINE COMPLETE


In [7]:
# ============================================================
# Payload + Runtime Footprint Check
# ============================================================

import requests
import json
import sys

API_BASE = "http://127.0.0.1:8000"
TEST_DATE = "2025-09-15"

endpoints = {
    "Forecast Grid": (
        "POST",
        "/forecast-grid",
        {"forecast_date": TEST_DATE},
    ),

    "Iceberg Risk Grid": (
        "POST",
        "/iceberg-risk-grid",
        {"forecast_date": TEST_DATE},
    ),

    "Route": (
        "POST",
        "/route",
        {
            "forecast_date": TEST_DATE,
            "vessel_profile": "standard",
            "start_latitude": -59.9258156,
            "start_longitude": 49.8904037,
            "destination_latitude": -59.9328194,
            "destination_longitude": 68.5594406,
        },
    ),
}

print("=" * 70)
print("API RESPONSE FOOTPRINT CHECK")
print("=" * 70)

for name, (method, path, payload) in endpoints.items():

    response = requests.request(
        method,
        API_BASE + path,
        json=payload,
        timeout=180,
    )

    print(f"\n{name}")
    print("HTTP:", response.status_code)

    if not response.ok:
        print("Request failed:")
        print(response.text[:500])
        continue

    print(
        "Response size:",
        f"{len(response.content) / 1024:.1f} KB"
    )

    try:
        data = response.json()

        print(
            "Top-level keys:",
            list(data.keys())
        )

        for key in [
            "points",
            "predicted_icebergs",
            "coordinates",
        ]:
            if key in data and isinstance(
                data[key],
                list
            ):
                print(
                    f"{key}:",
                    len(data[key]),
                    "items"
                )

    except Exception as exc:
        print(
            "JSON inspection error:",
            exc
        )

print("\n" + "=" * 70)
print("FOOTPRINT CHECK COMPLETE")
print("=" * 70)

API RESPONSE FOOTPRINT CHECK

Forecast Grid
HTTP: 200
Response size: 853.0 KB
Top-level keys: ['status', 'forecast_date', 'prediction_date', 'grid_step', 'valid_cells', 'predicted_sic', 'points']
points: 9584 items

Iceberg Risk Grid
HTTP: 200
Response size: 861.3 KB
Top-level keys: ['status', 'forecast_date', 'prediction_date', 'grid_step', 'iceberg_coverage', 'cost_parameters', 'predicted_icebergs', 'points']
points: 4651 items
predicted_icebergs: 31 items

Route
HTTP: 200
Response size: 5.3 KB
Top-level keys: ['status', 'start', 'destination', 'route', 'ice_exposure', 'coordinates', 'geometry', 'requested_start', 'requested_destination', 'grid_start', 'grid_goal', 'navigation_mode', 'decision', 'forecast', 'vessel', 'iceberg']
coordinates: 38 items

FOOTPRINT CHECK COMPLETE


In [8]:
# ============================================================
# Backend Stage Profiling
# ============================================================

import time
import numpy as np

from src.module1_forecast import forecast_sic
from src.module1_risk import classify_sic_risk
from src.vessel_profiles import create_vessel_cost_surface

from src.module2_trajectory import (
    get_coverage_aware_predictions
)

from src.module2_risk import (
    create_weighted_iceberg_cost_surface
)

from src.module3_decision import (
    create_integrated_navigation_cost
)

DATE = "2025-09-15"
PROFILE = "standard"

print("=" * 70)
print("ELSHADDAI BACKEND STAGE PROFILING")
print("=" * 70)


# ------------------------------------------------------------
# Module 1
# ------------------------------------------------------------

t0 = time.perf_counter()

forecast = forecast_sic(DATE)

t1 = time.perf_counter()

predicted_sic = forecast["predicted_sic"]

risk_code = classify_sic_risk(
    predicted_sic
)

t2 = time.perf_counter()

sea_ice_cost = create_vessel_cost_surface(
    risk_code,
    PROFILE
)

t3 = time.perf_counter()


print("\nMODULE 1")
print(
    f"forecast_sic:        {t1 - t0:.3f} s"
)
print(
    f"risk classification: {t2 - t1:.3f} s"
)
print(
    f"vessel cost surface: {t3 - t2:.3f} s"
)


# ------------------------------------------------------------
# Module 2 — iceberg predictions
# ------------------------------------------------------------

t4 = time.perf_counter()

coverage = get_coverage_aware_predictions(
    DATE,
    max_persistence_days=3,
    persistence_hazard_weight=0.25
)

t5 = time.perf_counter()

print("\nMODULE 2")
print(
    f"coverage/predictions: {t5 - t4:.3f} s"
)

print(
    "represented tracks:",
    coverage["coverage"]["represented_tracks"]
)


# ------------------------------------------------------------
# Module 2 — weighted cost surface
# ------------------------------------------------------------

t6 = time.perf_counter()

weighted = create_weighted_iceberg_cost_surface(
    forecast["latitude"],
    forecast["longitude"],
    coverage["predictions"],
    influence_km=30.0,
    hard_avoid_km=5.0,
    max_penalty=500.0
)

t7 = time.perf_counter()

iceberg_cost = weighted["iceberg_cost"]

print(
    f"weighted iceberg cost: {t7 - t6:.3f} s"
)


# ------------------------------------------------------------
# Module 3
# ------------------------------------------------------------

t8 = time.perf_counter()

navigation_cost = create_integrated_navigation_cost(
    sea_ice_cost=sea_ice_cost,
    iceberg_cost=iceberg_cost,
    iceberg_weight=1.0
)

t9 = time.perf_counter()

print("\nMODULE 3")
print(
    f"cost fusion:           {t9 - t8:.3f} s"
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

times = {
    "Module 1 forecast": t1 - t0,
    "Module 1 risk": t2 - t1,
    "Module 1 vessel cost": t3 - t2,
    "Module 2 predictions": t5 - t4,
    "Module 2 weighted iceberg cost": t7 - t6,
    "Module 3 fusion": t9 - t8,
}

print("\n" + "=" * 70)
print("STAGE TIMING SUMMARY")
print("=" * 70)

for name, seconds in times.items():
    print(
        f"{name:<35} {seconds:.3f} s"
    )

print("=" * 70)

ELSHADDAI BACKEND STAGE PROFILING

MODULE 1
forecast_sic:        5.012 s
risk classification: 0.003 s
vessel cost surface: 0.002 s

MODULE 2
coverage/predictions: 5.925 s
represented tracks: 31
weighted iceberg cost: 0.902 s

MODULE 3
cost fusion:           0.004 s

STAGE TIMING SUMMARY
Module 1 forecast                   5.012 s
Module 1 risk                       0.003 s
Module 1 vessel cost                0.002 s
Module 2 predictions                5.925 s
Module 2 weighted iceberg cost      0.902 s
Module 3 fusion                     0.004 s


In [9]:
# ============================================================
# Navigation Context Cache Test
# ============================================================

import time

from src.navigation_context import (
    get_navigation_context,
    clear_navigation_context_cache,
)

TEST_DATE = "2025-09-15"
TEST_PROFILE = "standard"

clear_navigation_context_cache()

print("=" * 70)
print("NAVIGATION CONTEXT CACHE TEST")
print("=" * 70)

# ------------------------------------------------------------
# First call — cache miss
# ------------------------------------------------------------

start = time.perf_counter()

context_1 = get_navigation_context(
    TEST_DATE,
    TEST_PROFILE,
)

first_time = time.perf_counter() - start


# ------------------------------------------------------------
# Second call — cache hit
# ------------------------------------------------------------

start = time.perf_counter()

context_2 = get_navigation_context(
    TEST_DATE,
    TEST_PROFILE,
)

second_time = time.perf_counter() - start


print("\nFirst call:")
print(f"  Time: {first_time:.3f} s")

print("\nSecond call:")
print(f"  Time: {second_time:.6f} s")

print("\nCache info:")
print(
    get_navigation_context.cache_info()
)

print("\nSame navigation cost object:",
      context_1["navigation_cost"] is
      context_2["navigation_cost"])

print("\nNavigation cost shape:",
      context_1["navigation_cost"].shape)

print("\nIceberg coverage:",
      context_1["iceberg_coverage"]["coverage"])


print("\n" + "=" * 70)

if (
    second_time < first_time * 0.1
    and
    context_1["navigation_cost"].shape
    == context_2["navigation_cost"].shape
):
    print("NAVIGATION CACHE TEST: PASS")
else:
    print("NAVIGATION CACHE TEST: REVIEW")

print("=" * 70)

NAVIGATION CONTEXT CACHE TEST

First call:
  Time: 10.988 s

Second call:
  Time: 0.000969 s

Cache info:
CacheInfo(hits=1, misses=1, maxsize=8, currsize=1)

Same navigation cost object: True

Navigation cost shape: (432, 432)

Iceberg coverage: {'total_tracks': 127, 'ml_predictions': 7, 'persistence_estimates': 24, 'represented_tracks': 31, 'excluded_tracks': 96, 'coverage_percent': 24.41}

NAVIGATION CACHE TEST: PASS


In [10]:
from pathlib import Path
import re

api_file = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
text = api_file.read_text(encoding="utf-8")

# Find the complete /route endpoint
pattern = re.compile(
    r'@app\.post\("/route"\).*?(?=\n# --------------------------------------------------\n# Module 4 — Dynamic route replanning)',
    re.DOTALL
)

match = pattern.search(text)

if not match:
    raise RuntimeError(
        "Could not locate the /route endpoint."
    )

new_route = r'''@app.post("/route")
def route(request: RouteRequest) -> Dict[str, Any]:
    """
    Generate a sea-ice + iceberg-aware navigation route.
    """

    try:

        # --------------------------------------------------
        # Shared cached navigation environment
        # --------------------------------------------------
        navigation_context = get_navigation_context(
            request.forecast_date,
            request.vessel_profile
        )

        forecast_result = (
            navigation_context["forecast_result"]
        )

        predicted_sic = (
            forecast_result["predicted_sic"]
        )

        risk_code = classify_sic_risk(
            predicted_sic
        )

        sea_ice_navigation_cost = (
            navigation_context["sea_ice_cost"]
        )

        iceberg_navigation_cost = (
            navigation_context["iceberg_cost"]
        )

        navigation_cost = (
            navigation_context["navigation_cost"]
        )

        iceberg_coverage = (
            navigation_context["iceberg_coverage"]
        )

        # --------------------------------------------------
        # Spatial output expected by route engine
        # --------------------------------------------------
        spatial_output = {
            "predicted_sic": predicted_sic,
            "risk_code": risk_code,
            "latitude": forecast_result["latitude"],
            "longitude": forecast_result["longitude"],
            "yc": forecast_result["yc"],
            "xc": forecast_result["xc"],
        }

        # --------------------------------------------------
        # Module 4 — Initial route
        # --------------------------------------------------
        result = plan_navigation_route(
            start_latitude=request.start_latitude,
            start_longitude=request.start_longitude,
            destination_latitude=request.destination_latitude,
            destination_longitude=request.destination_longitude,
            navigation_cost=navigation_cost,
            spatial_output=spatial_output
        )

        # --------------------------------------------------
        # Module 3 — Explain selected route
        # --------------------------------------------------
        route_coordinates = result.get(
            "coordinates",
            []
        )

        route_grid_cells = []

        for point in route_coordinates:

            if not isinstance(point, dict):
                continue

            route_latitude = point.get(
                "latitude"
            )

            route_longitude = point.get(
                "longitude"
            )

            if (
                route_latitude is None
                or route_longitude is None
            ):
                continue

            cell_distance = (
                np.abs(
                    forecast_result["latitude"]
                    - float(route_latitude)
                )
                +
                np.abs(
                    forecast_result["longitude"]
                    - float(route_longitude)
                )
            )

            nearest_cell = np.unravel_index(
                np.nanargmin(cell_distance),
                cell_distance.shape
            )

            route_grid_cells.append(
                nearest_cell
            )

        if route_grid_cells:

            decision = explain_route_decision(
                route_grid_cells,
                sea_ice_navigation_cost,
                iceberg_navigation_cost,
                iceberg_weight=1.0
            )

            result["decision"] = decision

        # --------------------------------------------------
        # Forecast information
        # --------------------------------------------------
        result["forecast"] = {
            "forecast_date": str(
                forecast_result["forecast_date"]
            ),
            "prediction_date": str(
                forecast_result["prediction_date"]
            )
        }

        # --------------------------------------------------
        # Vessel information
        # --------------------------------------------------
        result["vessel"] = {
            "profile": request.vessel_profile
        }

        # --------------------------------------------------
        # Module 2 information
        # --------------------------------------------------
        coverage = iceberg_coverage["coverage"]

        result["iceberg"] = {
            "total_tracks": coverage["total_tracks"],
            "ml_predictions": coverage["ml_predictions"],
            "persistence_estimates": coverage["persistence_estimates"],
            "represented_tracks": coverage["represented_tracks"],
            "excluded_tracks": coverage["excluded_tracks"],
            "coverage_percent": coverage["coverage_percent"],
            "influence_radius_km": 30.0,
            "hard_avoid_radius_km": 5.0,
            "max_penalty": 500.0
        }

        return result

    except ValueError as exc:

        raise HTTPException(
            status_code=400,
            detail=str(exc)
        )

    except FileNotFoundError as exc:

        raise HTTPException(
            status_code=503,
            detail=str(exc)
        )

    except RuntimeError as exc:

        raise HTTPException(
            status_code=422,
            detail=str(exc)
        )


'''

text = (
    text[:match.start()]
    + new_route
    + text[match.end():]
)

api_file.write_text(
    text,
    encoding="utf-8"
)

print("The /route endpoint was repaired successfully.")
print()
print("✓ Correct indentation restored")
print("✓ Cached navigation context retained")
print("✓ predicted_sic restored")
print("✓ Module 4 initial navigation retained")
print("✓ Module 3 decision explanation retained")
print("✓ Forecast/vessel/iceberg response fields retained")

The /route endpoint was repaired successfully.

✓ Correct indentation restored
✓ Cached navigation context retained
✓ predicted_sic restored
✓ Module 4 initial navigation retained
✓ Module 3 decision explanation retained
✓ Forecast/vessel/iceberg response fields retained


In [11]:
import subprocess
import sys
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")

result = subprocess.run(
    [
        sys.executable,
        "-c",
        "import src.api; print('API IMPORT: PASS')"
    ],
    cwd=str(project_root),
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

Return code: 0
API IMPORT: PASS



In [12]:
# ============================================================
# Route API Regression + Cache Performance Test
# ============================================================

import requests
import time

url = "http://127.0.0.1:8000/route"

payload = {
    "forecast_date": "2025-09-15",
    "vessel_profile": "standard",
    "start_latitude": -59.9258156,
    "start_longitude": 49.8904037,
    "destination_latitude": -59.9328194,
    "destination_longitude": 68.5594406,
}

print("=" * 70)
print("ROUTE API REGRESSION + CACHE TEST")
print("=" * 70)

times = []

for run in range(3):

    start = time.perf_counter()

    response = requests.post(
        url,
        json=payload,
        timeout=180,
    )

    elapsed = time.perf_counter() - start
    times.append(elapsed)

    print(
        f"Run {run + 1}: "
        f"{elapsed:.3f} s | "
        f"HTTP {response.status_code}"
    )

    if not response.ok:
        print(response.text)
        raise RuntimeError(
            "Route API request failed."
        )


data = response.json()

print("\nRoute regression:")
print(
    "  Navigation mode:",
    data.get("navigation_mode")
)

print(
    "  Distance:",
    data["route"]["distance_km"],
    "km"
)

print(
    "  Cost:",
    data["route"]["total_navigation_cost"]
)

print(
    "  Decision:",
    data.get("decision", {}).get(
        "dominant_hazard"
    )
)

print("\nPerformance:")
print(
    f"  First run : {times[0]:.3f} s"
)

print(
    f"  Later mean: "
    f"{sum(times[1:]) / len(times[1:]):.3f} s"
)

print("\n" + "=" * 70)
print("ROUTE API REGRESSION TEST: PASS")
print("=" * 70)

ROUTE API REGRESSION + CACHE TEST
Run 1: 10.335 s | HTTP 200
Run 2: 0.069 s | HTTP 200
Run 3: 0.064 s | HTTP 200

Route regression:
  Navigation mode: initial_route
  Distance: 1152.82 km
  Cost: 46.11
  Decision: sea_ice

Performance:
  First run : 10.335 s
  Later mean: 0.067 s

ROUTE API REGRESSION TEST: PASS


In [14]:
import requests

url = "http://127.0.0.1:8000/replan"

payload = {
    "forecast_date": "2025-09-20",
    "vessel_profile": "standard",
    "current_latitude": -59.7947654724,
    "current_longitude": 60.0684890747,
    "destination_latitude": -59.9328194,
    "destination_longitude": 68.5594406
}

response = requests.post(url, json=payload, timeout=60)

print("HTTP status:", response.status_code)

if response.status_code != 200:
    print(response.text)
else:
    result = response.json()

    print("Navigation mode:", result.get("navigation_mode"))
    print("Distance:", result.get("route", {}).get("distance_km"), "km")
    print("Cost:", result.get("route", {}).get("total_cost"))
    print("Decision:", result.get("decision", {}).get("dominant_hazard"))

    print("\nREPLAN REGRESSION: PASS")

HTTP status: 200
Navigation mode: replanned_route
Distance: 562.13 km
Cost: None
Decision: sea_ice

REPLAN REGRESSION: PASS


In [15]:
import requests
import json

url = "http://127.0.0.1:8000/replan"

payload = {
    "forecast_date": "2025-09-20",
    "vessel_profile": "standard",
    "current_latitude": -59.7947654724,
    "current_longitude": 60.0684890747,
    "destination_latitude": -59.9328194,
    "destination_longitude": 68.5594406
}

response = requests.post(url, json=payload, timeout=60)

print("HTTP:", response.status_code)

result = response.json()

print("\nTop-level keys:")
print(list(result.keys()))

print("\nRoute object:")
print(json.dumps(result.get("route"), indent=2))

HTTP: 200

Top-level keys:
['status', 'start', 'destination', 'route', 'ice_exposure', 'coordinates', 'geometry', 'requested_start', 'requested_destination', 'grid_start', 'grid_goal', 'navigation_mode', 'route_change', 'decision', 'forecast', 'vessel', 'current_position', 'iceberg']

Route object:
{
  "grid_cells": 21,
  "distance_km": 562.13,
  "straight_line_distance_km": 492.44,
  "detour_factor": 1.142,
  "extra_distance_km": 69.69,
  "total_navigation_cost": 35.99
}


In [16]:
import requests
import time

url = "http://127.0.0.1:8000/replan"

payload = {
    "forecast_date": "2025-09-20",
    "vessel_profile": "standard",
    "current_latitude": -59.7947654724,
    "current_longitude": 60.0684890747,
    "destination_latitude": -59.9328194,
    "destination_longitude": 68.5594406
}

times = []

print("=" * 70)
print("REPLAN API REGRESSION + CACHE TEST")
print("=" * 70)

for i in range(3):
    start = time.perf_counter()

    response = requests.post(
        url,
        json=payload,
        timeout=60
    )

    elapsed = time.perf_counter() - start
    times.append(elapsed)

    print(
        f"Run {i+1}: {elapsed:.3f} s | "
        f"HTTP {response.status_code}"
    )

    assert response.status_code == 200, response.text

result = response.json()

route = result["route"]

print("\nReplan regression:")
print(f"  Navigation mode: {result['navigation_mode']}")
print(f"  Distance: {route['distance_km']} km")
print(f"  Cost: {route['total_navigation_cost']}")
print(
    f"  Decision: "
    f"{result['decision']['dominant_hazard']}"
)

print("\nPerformance:")
print(f"  First run : {times[0]:.3f} s")
print(f"  Later mean: {sum(times[1:]) / 2:.3f} s")

assert result["navigation_mode"] == "replanned_route"
assert abs(route["distance_km"] - 562.13) < 0.1
assert abs(route["total_navigation_cost"] - 35.99) < 0.1

print("\n" + "=" * 70)
print("REPLAN API REGRESSION TEST: PASS")
print("=" * 70)

REPLAN API REGRESSION + CACHE TEST
Run 1: 0.066 s | HTTP 200
Run 2: 0.063 s | HTTP 200
Run 3: 0.061 s | HTTP 200

Replan regression:
  Navigation mode: replanned_route
  Distance: 562.13 km
  Cost: 35.99
  Decision: sea_ice

Performance:
  First run : 0.066 s
  Later mean: 0.062 s

REPLAN API REGRESSION TEST: PASS


In [17]:
from src.navigation_context import (
    get_navigation_context,
    clear_navigation_context_cache,
)

print("=" * 70)
print("NAVIGATION CONTEXT CACHE CORRECTNESS TEST")
print("=" * 70)

# Start clean
clear_navigation_context_cache()

# First context
ctx_a1 = get_navigation_context(
    "2025-09-15",
    "standard"
)

# Same request again
ctx_a2 = get_navigation_context(
    "2025-09-15",
    "standard"
)

# Different forecast date
ctx_b = get_navigation_context(
    "2025-09-20",
    "standard"
)

# Same date, different vessel profile
ctx_c = get_navigation_context(
    "2025-09-15",
    "ice_capable"
)

print("Same request returns same cached object:", ctx_a1 is ctx_a2)
print("Different date returns different object:", ctx_a1 is not ctx_b)
print("Different vessel profile returns different object:", ctx_a1 is not ctx_c)

print("\nCache info:")
print(get_navigation_context.cache_info())

assert ctx_a1 is ctx_a2
assert ctx_a1 is not ctx_b
assert ctx_a1 is not ctx_c

print("\n" + "=" * 70)
print("CACHE CORRECTNESS TEST: PASS")
print("=" * 70)

NAVIGATION CONTEXT CACHE CORRECTNESS TEST
Same request returns same cached object: True
Different date returns different object: True
Different vessel profile returns different object: True

Cache info:
CacheInfo(hits=1, misses=3, maxsize=8, currsize=3)

CACHE CORRECTNESS TEST: PASS


In [18]:
import requests
import time

print("=" * 70)
print("FINAL NAVIGATION API PERFORMANCE VALIDATION")
print("=" * 70)

route_url = "http://127.0.0.1:8000/route"
replan_url = "http://127.0.0.1:8000/replan"

route_payload = {
    "forecast_date": "2025-09-15",
    "vessel_profile": "standard",
    "start_latitude": -59.9258156,
    "start_longitude": 49.8904037,
    "destination_latitude": -59.9328194,
    "destination_longitude": 68.5594406
}

replan_payload = {
    "forecast_date": "2025-09-20",
    "vessel_profile": "standard",
    "current_latitude": -59.7947654724,
    "current_longitude": 60.0684890747,
    "destination_latitude": -59.9328194,
    "destination_longitude": 68.5594406
}


def benchmark(name, url, payload):
    times = []
    responses = []

    for _ in range(3):
        start = time.perf_counter()

        response = requests.post(
            url,
            json=payload,
            timeout=60
        )

        elapsed = time.perf_counter() - start

        times.append(elapsed)
        responses.append(response)

    result = responses[-1].json()

    print(f"\n{name}")
    print("-" * 70)
    print(f"HTTP statuses : {[r.status_code for r in responses]}")
    print(f"Run 1         : {times[0]:.3f} s")
    print(f"Later mean    : {sum(times[1:]) / 2:.3f} s")

    route = result["route"]

    print(f"Mode          : {result['navigation_mode']}")
    print(f"Distance      : {route['distance_km']} km")
    print(f"Navigation cost: {route['total_navigation_cost']}")
    print(
        f"Decision      : "
        f"{result['decision']['dominant_hazard']}"
    )

    assert all(r.status_code == 200 for r in responses)

    return result, times


route_result, route_times = benchmark(
    "INITIAL ROUTE (/route)",
    route_url,
    route_payload
)

replan_result, replan_times = benchmark(
    "DYNAMIC REPLAN (/replan)",
    replan_url,
    replan_payload
)

print("\n" + "=" * 70)
print("FINAL NAVIGATION API VALIDATION: PASS")
print("=" * 70)

FINAL NAVIGATION API PERFORMANCE VALIDATION

INITIAL ROUTE (/route)
----------------------------------------------------------------------
HTTP statuses : [200, 200, 200]
Run 1         : 10.887 s
Later mean    : 0.077 s
Mode          : initial_route
Distance      : 1152.82 km
Navigation cost: 46.11
Decision      : sea_ice

DYNAMIC REPLAN (/replan)
----------------------------------------------------------------------
HTTP statuses : [200, 200, 200]
Run 1         : 0.040 s
Later mean    : 0.038 s
Mode          : replanned_route
Distance      : 562.13 km
Navigation cost: 35.99
Decision      : sea_ice

FINAL NAVIGATION API VALIDATION: PASS
